In [ ]:
"""
================================================================================
LAB: Student Record Management System
File: Day1_Lab_RecordManager_completed.py
--------------------------------------------------------------------------------
Trainer Section (First 30%):
  - In-memory database initialization
  - File loading logic with JSON exception handling
  - View all records formatted output
  - Interactive CLI loop scaffold

Student Completion Tasks (Remaining 70%):
  1. Complete add_student_record() with input validation
  2. Implement search_student_record() by ID or Name substring
  3. Implement delete_student_record() with confirmation
  4. Implement update_student_record()
  5. Implement save_records_to_json() with safe file flushing
  6. STRETCH GOAL: Implement export_to_csv()
================================================================================
"""

import csv
import json
import os
import sys
from typing import Dict, Any

# Target data file
DATABASE_FILE = "sample_records.json"

# In-memory storage: Key = Student ID, Value = Dict of Student attributes
STUDENT_REGISTRY: Dict[str, Dict[str, Any]] = {}


def load_records_from_json(file_path: str) -> Dict[str, Dict[str, Any]]:
    """Loads student records safely from a JSON file.

    Handles FileNotFoundError and corrupted JSON formatting gracefully.
    """
    if not os.path.exists(file_path):
        print(f"[WARN] Database file '{file_path}' not found. Starting with empty registry.")
        return {}

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)
            print(f"[SUCCESS] Loaded {len(data)} record(s) from {file_path}.")
            return data
    except json.JSONDecodeError as json_err:
        print(f"[ERROR] Corrupted JSON structure in '{file_path}': {json_err}")
        return {}
    except Exception as err:
        print(f"[UNEXPECTED ERROR] Failed to load data: {err}")
        return {}


def view_all_records(registry: Dict[str, Dict[str, Any]]) -> None:
    """Prints all student records in a formatted tabular view."""
    if not registry:
        print("\n[INFO] No records found in the registry.")
        return

    separator = "-" * 75
    print("\n" + separator)
    print(f"{'Student ID':<12} | {'Name':<22} | {'Branch':<22} | {'CGPA':<5}")
    print(separator)
    for student_id, details in registry.items():
        name = details.get("name", "N/A")
        branch = details.get("branch", "N/A")
        cgpa = details.get("cgpa", 0.0)
        print(f"{student_id:<12} | {name:<22} | {branch:<22} | {cgpa:<5.2f}")
    print(separator + "\n")


# ==============================================================================
# ✍️ STUDENT TASKS (IMPLEMENTED)
# ==============================================================================

def add_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """Task 1: Prompt user for student ID, name, branch, and CGPA.

    Validation Rules:
      1. Student ID must not already exist in the registry.
      2. Name and Branch must not be empty after stripping whitespace.
      3. CGPA must be a valid float between 0.0 and 10.0.
      4. Auto-generate email: <first_name_lowercase>.<id_lowercase>@university.edu
    """
    print("\n--- Add New Student ---")

    # --- Student ID ---
    student_id = input("Enter Student ID: ").strip()
    if not student_id:
        print("[ERROR] Student ID cannot be empty. Aborting.")
        return
    if student_id in registry:
        print(f"[ERROR] Student ID '{student_id}' already exists. Aborting.")
        return

    # --- Name ---
    name = input("Enter Name: ").strip()
    if not name:
        print("[ERROR] Name cannot be empty. Aborting.")
        return

    # --- Branch ---
    branch = input("Enter Branch: ").strip()
    if not branch:
        print("[ERROR] Branch cannot be empty. Aborting.")
        return

    # --- CGPA ---
    cgpa_raw = input("Enter CGPA (0.0 - 10.0): ").strip()
    try:
        cgpa = float(cgpa_raw)
    except ValueError:
        print(f"[ERROR] '{cgpa_raw}' is not a valid number. Aborting.")
        return
    if not (0.0 <= cgpa <= 10.0):
        print("[ERROR] CGPA must be between 0.0 and 10.0. Aborting.")
        return

    # --- Auto-generate email ---
    first_name = name.split()[0].lower()
    email = f"{first_name}.{student_id.lower()}@university.edu"

    registry[student_id] = {
        "name": name,
        "branch": branch,
        "cgpa": cgpa,
        "email": email,
    }

    print(f"[SUCCESS] Added student '{name}' (ID: {student_id}) with email {email}.")


def search_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """Task 2: Search by student ID (exact) or student Name (case-insensitive substring)."""
    print("\n--- Search Student Records ---")

    if not registry:
        print("[INFO] Registry is empty. Nothing to search.")
        return

    query = input("Enter Student ID or Name (substring) to search: ").strip()
    if not query:
        print("[ERROR] Search query cannot be empty.")
        return

    # 1. Exact ID match
    if query in registry:
        details = registry[query]
        print(f"\n[FOUND] Exact ID match:")
        print(f"  ID: {query}")
        for key, value in details.items():
            print(f"  {key.capitalize()}: {value}")
        return

    # 2. Case-insensitive substring match on name
    query_lower = query.lower()
    matches = {
        sid: details
        for sid, details in registry.items()
        if query_lower in details.get("name", "").lower()
    }

    if not matches:
        print(f"[INFO] No records found matching '{query}'.")
        return

    print(f"\n[FOUND] {len(matches)} record(s) matching name '{query}':")
    for sid, details in matches.items():
        print(f"  ID: {sid} | Name: {details.get('name', 'N/A')} | "
              f"Branch: {details.get('branch', 'N/A')} | CGPA: {details.get('cgpa', 0.0):.2f}")


def delete_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """Task 3: Prompt for Student ID and delete record with confirmation."""
    print("\n--- Delete Student Record ---")

    student_id = input("Enter Student ID to delete: ").strip()
    if student_id not in registry:
        print(f"[ERROR] Student ID '{student_id}' not found in registry.")
        return

    name = registry[student_id].get("name", "N/A")
    confirm = input(f"Are you sure you want to delete '{name}' (ID: {student_id})? [y/N]: ").strip().lower()

    if confirm == "y":
        del registry[student_id]
        print(f"[SUCCESS] Deleted record for '{name}' (ID: {student_id}).")
    else:
        print("[INFO] Deletion cancelled.")


def update_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """Update an existing student's branch and/or CGPA.

    Blank input leaves the existing field unchanged.
    """
    print("\n--- Update Student Record ---")

    student_id = input("Enter Student ID to update: ").strip()
    if student_id not in registry:
        print(f"[ERROR] Student ID '{student_id}' not found in registry.")
        return

    details = registry[student_id]
    print(f"Leave a field blank to keep its current value.")

    new_branch = input(f"Enter new Branch [{details.get('branch', 'N/A')}]: ").strip()
    if new_branch:
        details["branch"] = new_branch

    new_cgpa_raw = input(f"Enter new CGPA [{details.get('cgpa', 0.0)}]: ").strip()
    if new_cgpa_raw:
        try:
            new_cgpa = float(new_cgpa_raw)
            if 0.0 <= new_cgpa <= 10.0:
                details["cgpa"] = new_cgpa
            else:
                print("[ERROR] CGPA must be between 0.0 and 10.0. Skipping CGPA update.")
        except ValueError:
            print(f"[ERROR] '{new_cgpa_raw}' is not a valid number. Skipping CGPA update.")

    print(f"[SUCCESS] Updated record for Student ID '{student_id}'.")


def save_records_to_json(file_path: str, registry: Dict[str, Dict[str, Any]]) -> None:
    """Task 4: Serialize the in-memory registry dictionary to the JSON file safely."""
    try:
        with open(file_path, "w", encoding="utf-8") as file:
            json.dump(registry, file, indent=2)
            file.flush()
            os.fsync(file.fileno())
        print(f"[SUCCESS] Saved {len(registry)} record(s) to '{file_path}'.")
    except Exception as err:
        print(f"[ERROR] Failed to save data to '{file_path}': {err}")


def export_to_csv(file_path: str, registry: Dict[str, Dict[str, Any]]) -> None:
    """Task 5 (Bonus): Export all student records to a CSV file."""
    if not registry:
        print("[INFO] Registry is empty. Nothing to export.")
        return

    fieldnames = ["student_id", "name", "branch", "cgpa", "email"]

    try:
        with open(file_path, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            for student_id, details in registry.items():
                writer.writerow({
                    "student_id": student_id,
                    "name": details.get("name", ""),
                    "branch": details.get("branch", ""),
                    "cgpa": details.get("cgpa", 0.0),
                    "email": details.get("email", ""),
                })
        print(f"[SUCCESS] Exported {len(registry)} record(s) to '{file_path}'.")
    except Exception as err:
        print(f"[ERROR] Failed to export data to '{file_path}': {err}")


def main_menu() -> None:
    """Main CLI control loop."""
    global STUDENT_REGISTRY
    STUDENT_REGISTRY = load_records_from_json(DATABASE_FILE)

    menu_banner = """
========================================
🎓 STUDENT RECORD MANAGEMENT SYSTEM
========================================
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
7. Update Student Record
0. Save & Exit
========================================
"""
    while True:
        print(menu_banner)
        choice = input("Enter choice [0-7]: ").strip()

        if choice == "1":
            view_all_records(STUDENT_REGISTRY)
        elif choice == "2":
            add_student_record(STUDENT_REGISTRY)
        elif choice == "3":
            search_student_record(STUDENT_REGISTRY)
        elif choice == "4":
            delete_student_record(STUDENT_REGISTRY)
        elif choice == "5":
            save_records_to_json(DATABASE_FILE, STUDENT_REGISTRY)
        elif choice == "6":
            export_to_csv("students_export.csv", STUDENT_REGISTRY)
        elif choice == "7":
            update_student_record(STUDENT_REGISTRY)
        elif choice == "0":
            save_records_to_json(DATABASE_FILE, STUDENT_REGISTRY)
            print("[INFO] Application closed successfully. Good bye!")
            sys.exit(0)
        else:
            print("[WARN] Invalid option selected. Please enter a number between 0 and 7.")


if __name__ == "__main__":
    main_menu()


[SUCCESS] Loaded 3 record(s) from sample_records.json.

🎓 STUDENT RECORD MANAGEMENT SYSTEM
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
7. Update Student Record
0. Save & Exit


---------------------------------------------------------------------------
Student ID   | Name                   | Branch                 | CGPA 
---------------------------------------------------------------------------
STU-1001     | Sarah Jenkins          | Computer Science       | 9.20 
STU-1002     | Marcus Vance           | Mechanical Engineering | 7.80 
STU-1003     | Priya Sharma           | Information Technology | 8.90 
---------------------------------------------------------------------------


🎓 STUDENT RECORD MANAGEMENT SYSTEM
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
7. Update Student Record
0. Save & Exit


--------